# Collaborative Filtering for Implicit Feedback Datasets 

*Разбор статьи: Yifan Hu, Yehuda Koren, Chris Volinsky. "Collaborative Filtering for Implicit Feedback Datasets", IEEE ICDM 2008.*

Эта статья — одна из самых цитируемых работ в рекомендательных системах. Именно она лежит в основе алгоритма **implicit ALS**, который используется в библиотеке `implicit` и во множестве промышленных рекомендательных систем. Ниже — пошаговый разбор идей статьи простым языком, с пояснением всех формул.

---

## 1. О чём вообще статья и зачем она нужна

Представим интернет-магазин или онлайн-кинотеатр, который хочет порекомендовать пользователю что-то интересное. Есть два принципиально разных источника информации о вкусах пользователя:

- **Explicit feedback (явная обратная связь)** — пользователь сам, осознанно, сообщает своё мнение: ставит фильму оценку 4 из 5 звёзд, нажимает «лайк» или «дизлайк».
- **Implicit feedback (неявная обратная связь)** — мы ничего не спрашиваем напрямую, а просто **наблюдаем** за поведением: что пользователь купил, что посмотрел, на что кликнул, сколько времени провёл на странице.

Проблема в том, что явных оценок обычно **очень мало** — большинство людей никогда не ставят звёздочки фильмам. А вот неявных сигналов (кликов, просмотров, покупок) — очень много, они собираются автоматически, без каких-либо усилий со стороны пользователя.

Авторы статьи работали над рекомендательной системой для **телевизионных передач** (по данным приставок цифрового телевидения) и у них не было явных оценок вообще — только то, какие каналы и когда смотрел пользователь. Поэтому им пришлось придумать способ строить качественные рекомендации, опираясь исключительно на implicit-сигналы.

Главный вклад статьи:
1. Авторы чётко формулируют, **чем implicit-данные принципиально отличаются** от explicit-данных, и почему старые алгоритмы для explicit-фидбэка нельзя просто взять и применить к implicit-данным.
2. Предлагают **новую модель** (по сути — вариант матричной факторизации), специально приспособленную под implicit-фидбэк.
3. Предлагают **эффективный алгоритм обучения**, который работает за время, линейно растущее с объёмом данных (а не квадратично или хуже) — это критично, ведь в implicit-постановке нужно учитывать *все* пары пользователь–айтем, а не только небольшое число известных оценок.
4. Показывают, как для такой модели можно **объяснять** рекомендации пользователю (то есть отвечать на вопрос «почему мне это порекомендовали?»).

---

## 2. Почему implicit-данные — это не то же самое, что explicit

Авторы выделяют четыре ключевых отличия implicit-фидбэка от explicit. Это самая важная концептуальная часть статьи — на ней строится всё остальное.

### 2.1. Нет негативной обратной связи

Если пользователь поставил фильму 1 звезду — это чёткий сигнал «не понравилось». А если пользователь **не посмотрел** какой-то фильм — что это значит? Может быть:
- фильм ему не нравится,
- он просто о нём не знал,
- у него не было возможности его посмотреть (например, был занят),
- фильм появился слишком недавно.

То есть **отсутствие действия — это не то же самое, что отрицательная оценка**. В explicit-данных мы точно знаем, что пользователю нравится, а что нет. В implicit-данных мы знаем только, с чем пользователь взаимодействовал — а про всё остальное можем лишь строить предположения.

Из-за этого в explicit-моделях обычно **игнорируют** пары «пользователь–айтем», для которых нет оценки (считают их «пропущенными данными», missing data). В implicit-постановке так делать нельзя: если анализировать только то, что пользователь посмотрел/купил, мы увидим исключительно «позитивные» примеры и получим сильно искажённую картину его вкусов. Приходится как-то учитывать и то, с чем пользователь **не** взаимодействовал.

### 2.2. Implicit-сигнал зашумлён

Даже если мы видим, что пользователь что-то купил, это не обязательно значит, что товар ему понравился — возможно, это был подарок, или покупка оказалась неудачной. Если мы видим, что телевизор был включён на определённом канале — возможно, зритель в этот момент просто спал.

### 2.3. Число (частота) implicit-сигнала означает уверенность, а не силу предпочтения

В explicit-данных значение — это сама оценка: 5 звёзд лучше, чем 1 звезда, это прямая шкала предпочтения.

В implicit-данных число обычно означает **частоту действия**: сколько раз пользователь посмотрел передачу, сколько раз купил товар. Но частое действие — это не всегда «более любимое» действие! Пример из статьи: любимый пользователем фильм он может посмотреть один единственный раз, а сериал, который ему просто нравится (не «любимый»), он смотрит каждую неделю, и число просмотров у сериала будет намного больше.

Тем не менее, число повторений всё равно полезно — оно говорит о том, **насколько мы уверены**, что это взаимодействие отражает реальный интерес пользователя. Одно случайное действие могло произойти по любой причине и ничего не говорить о вкусах, а повторяющееся действие с гораздо большей вероятностью отражает настоящий интерес.

### 2.4. Нужны специальные метрики оценки качества

В explicit-задаче легко посчитать, например, среднеквадратичную ошибку (MSE) между предсказанной и настоящей оценкой. С implicit-данными всё сложнее: нужно учитывать доступность товара (была ли у пользователя вообще возможность его купить), конкуренцию между товарами за внимание пользователя (если по телевизору в одно время идут два интересных шоу — оба посмотреть нельзя) и повторные действия.

---

## 3. Как раньше решали задачу CF (Collaborative Filtering)

Прежде чем перейти к предложенной авторами модели, вспомним, какие подходы существовали до этого.

### 3.1. Neighborhood-модели (модели соседей)

Идея простая и интуитивная: чтобы понять, понравится ли пользователю $u$ айтем $i$, найдём айтемы, **похожие** на $i$, которые пользователь $u$ уже оценивал, и усредним его оценки этих похожих айтемов (взвешенно, по степени похожести):

$$
\hat r_{ui} = \frac{\sum_{j \in S^k(i;u)} s_{ij}\, r_{uj}}{\sum_{j \in S^k(i;u)} s_{ij}}
$$

Здесь:
- $S^k(i;u)$ — множество из $k$ айтемов, наиболее похожих на $i$, которые пользователь $u$ уже оценил;
- $s_{ij}$ — мера схожести айтемов $i$ и $j$ (например, коэффициент корреляции Пирсона между тем, как разные пользователи оценивали эти два айтема);
- $r_{uj}$ — известная оценка пользователя $u$ айтему $j$.

По сути, формула — это **взвешенное среднее** известных оценок пользователя по похожим айтемам, где вес — степень похожести.

Такой подход хорошо интерпретируется («мы рекомендуем это, потому что вам понравилось похожее»), но у него есть существенный минус применительно к implicit-данным: он **не умеет различать** «пользователь действительно предпочитает этот айтем» и «мы просто не уверены в его предпочтениях» — то есть нет механизма явно учитывать уровень уверенности в наблюдении.

### 3.2. Latent factor models (модели скрытых факторов)

Альтернативный подход — не искать явное сходство между конкретными айтемами, а **выучить компактное числовое представление** (вектор) для каждого пользователя и каждого айтема так, чтобы скалярное произведение этих векторов предсказывало взаимодействие:

$$
\hat r_{ui} = x_u^{\mathsf T} y_i
$$

где $x_u \in \mathbb{R}^f$ — вектор («факторы») пользователя $u$, а $y_i \in \mathbb{R}^f$ — вектор айтема $i$, $f$ — количество факторов (размерность скрытого пространства).

Для explicit-данных (например, рейтингов Netflix) типичная функция потерь выглядит так:

$$
\min_{x_*, y_*} \sum_{r_{ui}\text{ известно}} \big(r_{ui} - x_u^{\mathsf T} y_i\big)^2 + \lambda\Big(\lVert x_u \rVert^2 + \lVert y_i \rVert^2\Big) \qquad (2)
$$

То есть мы минимизируем ошибку предсказания **только по тем парам, где оценка реально известна**, а слагаемое с $\lambda$ — это регуляризация, которая не даёт модели переобучиться (без неё модель могла бы просто «запомнить» все известные оценки, не научившись ничему полезному для новых пар).

На тот момент (конец 2000-х, эпоха соревнования Netflix Prize) такие модели, основанные на SVD (сингулярном разложении матрицы), стабильно показывали лучшее качество, чем модели соседей — но были разработаны для explicit-данных. Авторы статьи берут именно эту идею (латентные факторы) и адаптируют её под implicit-постановку.

---

## 4. Предложенная модель: разделяем «предпочтение» и «уверенность»

Это центральная идея всей статьи.

### 4.1. От сырых наблюдений — к предпочтению и уверенности

Пусть $r_{ui}$ — «сырое» наблюдение: например, сколько раз пользователь $u$ посмотрел передачу $i$ (0, если не смотрел вообще, 0.7 — если посмотрел 70% один раз, 2 — если посмотрел дважды и т.д.).

Авторы предлагают ввести **две отдельные величины** вместо одной:

**1) Бинарное предпочтение** $p_{ui}$ — было ли вообще какое-то взаимодействие:

$$
p_{ui} =
\begin{cases}
1, & r_{ui} > 0 \\
0, & r_{ui} = 0
\end{cases}
$$

Если пользователь хоть раз посмотрел передачу — считаем, что она ему, скорее всего, нравится ($p_{ui}=1$). Если не смотрел совсем — по умолчанию считаем, что предпочтения нет ($p_{ui}=0$), хотя мы понимаем, что это предположение может быть неверным (см. пункт 2.1 выше).

**2) Уверенность** $c_{ui}$ — насколько мы доверяем значению $p_{ui}$:

$$
c_{ui} = 1 + \alpha\, r_{ui}
$$

Это ключевая формула модели. Разберём её по частям:
- Даже когда $r_{ui} = 0$ (взаимодействия не было), уверенность равна $c_{ui} = 1$ — то есть **минимальная, но не нулевая** уверенность. Модель всё равно "видит" эту пару (пользователь, айтем), просто придаёт ей маленький вес.
- Чем больше $r_{ui}$ (то есть чем чаще/дольше пользователь взаимодействовал с айтемом), тем выше уверенность $c_{ui}$ — растёт линейно с коэффициентом $\alpha$.
- $\alpha$ — гиперпараметр, который регулирует, насколько быстро растёт уверенность. В экспериментах авторов хорошо сработало значение $\alpha = 40$.

Также авторы упоминают альтернативный (логарифмический) вариант этой формулы, полезный, когда значения $r_{ui}$ сильно различаются по масштабу (например, кто-то посмотрел передачу 1 раз, а кто-то — 200 раз за счёт записи на видеомагнитофон):

$$
c_{ui} = 1 + \alpha \log\!\big(1 + r_{ui}/\epsilon\big) \qquad (6)
$$

**Итог**: вместо одного числа $r_{ui}$ мы теперь работаем с парой чисел $(p_{ui}, c_{ui})$ — «что мы думаем о предпочтении» и «насколько мы в этом уверены». Это разделение и есть главная идея статьи.

### 4.2. Функция потерь

Теперь, как и в обычной матричной факторизации, ищем векторы $x_u$ (пользователи) и $y_i$ (айтемы), но с учётом уверенности $c_{ui}$ как веса ошибки:

$$
\min_{x_*, y_*} \sum_{u,i} c_{ui}\,\big(p_{ui} - x_u^{\mathsf T} y_i\big)^2 \;+\; \lambda\Big(\sum_u \lVert x_u \rVert^2 + \sum_i \lVert y_i \rVert^2\Big) \qquad (3)
$$

Сравним с формулой (2) для explicit-данных, и увидим два принципиальных отличия:

1. **Сумма теперь идёт по всем парам $(u, i)$** — то есть по всем пользователям и всем айтемам, а не только по тем парам, для которых наблюдение было известно. Это логично: раз мы считаем, что «отсутствие взаимодействия» — это тоже информация (пусть и неуверенная), мы обязаны учитывать *все* такие пары в обучении, а не выбрасывать их.
2. **У каждого слагаемого есть вес** $c_{ui}$ — ошибка на паре, в которой мы уверены (было много взаимодействий), штрафуется сильнее, а ошибка на паре с низкой уверенностью (взаимодействия не было) штрафуется слабо. Это позволяет модели «не бояться» ошибаться на многочисленных парах с нулевым предпочтением — ведь мы и сами не очень уверены, что там действительно ноль.

### 4.3. Главная вычислительная проблема

Формула (3) выглядит просто, но здесь скрыта серьёзная практическая проблема: сумма берётся по **всем** парам пользователь-айтем, а их — $n \times m$ (число пользователей, умноженное на число айтемов). Для реальных данных это может быть несколько миллиардов слагаемых! Из-за этого нельзя напрямую применить стохастический градиентный спуск (который отлично работал для explicit-данных, где известных пар было относительно немного).

---

## 5. Алгоритм оптимизации: Alternating Least Squares (ALS)

### 5.1. Общая идея ALS

Задача (3) — не выпуклая, если оптимизировать $x_u$ и $y_i$ одновременно (это произведение двух наборов неизвестных). **Но** если зафиксировать все векторы айтемов $y_i$ и оптимизировать только векторы пользователей $x_u$ — задача становится обычной (взвешенной) линейной регрессией, у которой есть точное аналитическое решение! И наоборот: если зафиксировать $x_u$, легко найти оптимальные $y_i$.

Отсюда и название метода: мы **чередуем** (alternate) шаги — то фиксируем айтемы и пересчитываем пользователей, то наоборот, и на каждом шаге ошибка гарантированно не увеличивается.

### 5.2. Аналитическая формула для пересчёта $x_u$

Соберём все векторы айтемов в матрицу $Y$ размера $n \times f$ (n — число айтемов, f — число факторов). Для конкретного пользователя $u$:

- $C^u$ — диагональная матрица размера $n \times n$, на диагонали которой стоят значения уверенности $c_{ui}$ для всех айтемов $i$;
- $p^{(u)}$ — вектор, содержащий все значения предпочтений $p_{ui}$ пользователя $u$ по всем айтемам.

Взяв производную функции потерь по $x_u$ и приравняв к нулю, получаем точное решение:

$$
x_u = \big(Y^{\mathsf T} C^u Y + \lambda I\big)^{-1} Y^{\mathsf T} C^u\, p^{(u)} \qquad (4)
$$

Симметрично, зафиксировав $x_*$, для каждого айтема:

$$
y_i = \big(X^{\mathsf T} C^i X + \lambda I\big)^{-1} X^{\mathsf T} C^i\, p^{(i)} \qquad (5)
$$

### 5.3. Как избежать «взрыва» вычислений — ключевой трюк статьи

Если считать формулу (4) в лоб, для **каждого** пользователя $u$ нужно перемножать матрицы размера $n \times n$ (где $n$ — число айтемов, может быть десятки тысяч) — это слишком медленно, если повторять для миллионов пользователей.

Авторы замечают важный факт: матрица $C^u - I$ (уверенность минус единица) содержит **не нули только там, где пользователь реально взаимодействовал с айтемом** — ведь для всех остальных айтемов $c_{ui} = 1$, поэтому $c_{ui} - 1 = 0$. А число айтемов, с которыми конкретный пользователь взаимодействовал ($n_u$), обычно **гораздо меньше** общего числа айтемов $n$.

Используя тождество:

$$
Y^{\mathsf T} C^u Y = Y^{\mathsf T} Y + Y^{\mathsf T}(C^u - I) Y
$$

замечаем, что:
- $Y^{\mathsf T}Y$ — общая для **всех** пользователей матрица размера $f \times f$, её достаточно посчитать **один раз** за $O(f^2 n)$;
- $Y^{\mathsf T}(C^u - I) Y$ — считается только по ненулевым элементам $C^u - I$, то есть за время, пропорциональное числу реальных взаимодействий пользователя $n_u$, а не полному числу айтемов $n$.

В итоге пересчёт одного $x_u$ занимает $O(f^2 n_u + f^3)$ вместо $O(f^2 n)$, а суммарное время по всем пользователям — $O(f^2 N + f^3 m)$, где $N$ — общее число реальных (ненулевых) наблюдений, а $m$ — число пользователей. Это **линейно** зависит от размера входных данных — именно это авторы и обещали во введении статьи.

Аналогичный трюк применяется и при пересчёте $y_i$ (айтемов), с использованием общей для всех айтемов матрицы $X^{\mathsf T}X$.

### 5.4. Алгоритм целиком

```
1. Инициализировать X (пользователи) и Y (айтемы) случайными значениями
2. Посчитать Y^T Y
3. Для каждого пользователя u: пересчитать x_u по формуле (4),
   используя предвычисленную Y^T Y и только ненулевые взаимодействия пользователя
4. Посчитать X^T X
5. Для каждого айтема i: пересчитать y_i по формуле (5),
   используя предвычисленную X^T X и только ненулевые взаимодействия айтема
6. Повторять шаги 2–5, пока метрики не стабилизируются (обычно ~10 проходов)
```

Важное практическое следствие: пересчёт $x_u$ для разных пользователей **не зависит друг от друга** (все используют одну и ту же предвычисленную $Y^{\mathsf T}Y$), поэтому этот шаг идеально распараллеливается — можно считать вектора всех пользователей одновременно на разных ядрах/машинах. То же самое верно и для айтемов.

---

## 6. Объяснение рекомендаций

Одна из проблем latent factor моделей (по сравнению с neighborhood-моделями) — их сложно **объяснить**: рекомендация получается из скалярного произведения двух абстрактных векторов, и непонятно, что именно из истории пользователя привело к такой рекомендации.

Авторы предлагают красивый трюк: подставим формулу (4) обратно в предсказание $\hat p_{ui} = y_i^{\mathsf T} x_u$:

$$
\hat p_{ui} = y_i^{\mathsf T} \big(Y^{\mathsf T} C^u Y + \lambda I\big)^{-1} Y^{\mathsf T} C^u\, p^{(u)}
$$

Обозначим $W^u = \big(Y^{\mathsf T} C^u Y + \lambda I\big)^{-1}$ (это матрица весов, своя для каждого пользователя) и $s^u_{ij} = y_i^{\mathsf T} W^u y_j$ («взвешенная похожесть» айтемов $i$ и $j$ с точки зрения конкретного пользователя $u$). Тогда предсказание переписывается как:

$$
\hat p_{ui} = \sum_{j:\, r_{uj} > 0} s^u_{ij}\, c_{uj} \qquad (7)
$$

Это очень наглядная формула: предсказанное предпочтение к айтему $i$ раскладывается в **сумму вкладов** от каждого айтема $j$, с которым пользователь реально взаимодействовал в прошлом! Вклад каждого прошлого действия зависит от:
- насколько мы были уверены в этом действии ($c_{uj}$),
- насколько айтем $j$ похож на рекомендуемый айтем $i$ с точки зрения этого пользователя ($s^u_{ij}$).

Таким образом latent factor модель по сути "превращается" обратно в понятную neighborhood-модель именно в момент объяснения — но при этом веса схожести $s^u_{ij}$ выучены гораздо более принципиальным способом (через общую оптимизацию), а не заданы вручную (например, через корреляцию Пирсона).

Практически это позволяет показать пользователю: «мы порекомендовали вам шоу X, потому что вы смотрели передачи A, B, C» — беря те несколько прошлых действий, у которых наибольший вклад в сумму (7). В примере из статьи: реалити-шоу *So You Think You Can Dance* объясняется другими реалити-шоу, *Spider-Man* — другими комикс-передачами, а *Life in the E.R.* — медицинскими документальными фильмами. Такие объяснения интуитивно понятны и повышают доверие пользователя к системе.

---

## 7. Как измеряли качество (эксперименты)

### 7.1. Данные

Данные собраны с примерно 300 000 телевизионных приставок за 4 недели (обучающая выборка) плюс ещё неделя (тестовая выборка). Значение $r_{ui}$ — сколько раз/насколько полно пользователь $u$ посмотрел передачу $i$ (например, 0.7 означает «посмотрел 70% передачи один раз»).

Авторы также вносят два практических уточнения в сырые данные:
- **Логарифмическое масштабирование** значений $r_{ui}$ (формула 6), так как они сильно различаются по величине (от долей единицы до сотен — например, если передача записывалась на видеомагнитофон и «просматривалась» много часов подряд).
- **Эффект инерции (momentum effect)**: если пользователь долго не переключал канал, вероятно, ему был интересен только первый показанный фрагмент, а дальше телевизор мог быть просто включён фоном (или зритель уснул!). Поэтому вклад последующих передач после переключения канала специально уменьшается по формуле, зависящей от порядкового номера передачи после переключения.

### 7.2. Как оценивали качество: метрика `rank`

Поскольку у нас нет надёжной негативной обратной связи (мы не знаем точно, какие передачи пользователю **не** нравятся), классические precision-метрики здесь плохо подходят. Вместо этого авторы используют метрику, основанную на **ранге** (позиции) реально просмотренных передач в списке рекомендаций:

- Для каждого пользователя строится полный отсортированный список всех передач — от наиболее рекомендуемой до наименее рекомендуемой.
- $rank_{ui}$ — процентильная позиция передачи $i$ в этом списке для пользователя $u$: $0\%$ означает «самая первая, наиболее рекомендуемая», $100\%$ — «самая последняя».
- Итоговая метрика — средневзвешенный (по фактическому времени просмотра $r^t_{ui}$) ранг реально просмотренных в тесте передач:

$$
\overline{rank} = \frac{\sum_{u,i} r^t_{ui}\, rank_{ui}}{\sum_{u,i} r^t_{ui}} \qquad (8)
$$

Чем **ниже** значение $\overline{rank}$, тем лучше модель — это значит, что реально просмотренные передачи модель ставила близко к верху списка рекомендаций. Для случайных (нерелевантных) рекомендаций ожидаемое значение — $50\%$ (передача в среднем оказывается в середине списка). Значит, любой разумный алгоритм должен показывать $\overline{rank}$ существенно меньше $50\%$.

### 7.3. Сравнение с бейзлайнами

Авторы сравнивают свою модель с двумя более простыми подходами:

1. **Popularity baseline** — просто рекомендовать всем самые популярные передачи (без всякой персонализации). Несмотря на примитивность, это на удивление сильный бейзлайн: люди действительно склонны смотреть то, что смотрят многие. Результат: $\overline{rank} = 16.46\%$.
2. **Neighborhood-модель** (item-item, на косинусной похожести) — более сильный бейзлайн, использующий персонализацию через похожие передачи. Результат: $\overline{rank} = 10.74\%$.
3. **Предложенная факторная модель** — лучший результат, $\overline{rank} = 8.35\%$ при 200 факторах, и качество продолжает расти с увеличением числа факторов (то есть чем больше размерность латентного пространства, тем точнее модель — в пределах вычислительных возможностей).

Также авторы проверяют **важность** самой идеи «предпочтение + уверенность», сравнивая три варианта модели:
- напрямую факторизовать сырые $r_{ui}$ (без разделения на $p_{ui}$/$c_{ui}$, формула 9) → $\overline{rank} \approx 13.4\%$ (плохо);
- факторизовать только бинарное предпочтение $p_{ui}$, без уверенности $c_{ui}$ (формула 10) → $\overline{rank} \approx 10.5\%$ (лучше, но всё ещё хуже полной модели);
- полная модель с $p_{ui}$ **и** $c_{ui}$ (формула 3) → $\overline{rank} \approx 8.6$–$8.9\%$ (лучший результат).

Это подтверждает главный тезис статьи: именно **разделение** сырого сигнала на предпочтение и уверенность даёт основной прирост качества, а не просто переход к латентным факторам как таковым.

### 7.4. Дополнительные наблюдения

- Модель гораздо лучше предсказывает просмотр **популярных** передач, чем непопулярных (из-за того, что по популярным передачам просто больше данных для обучения).
- В отличие от опыта с explicit-датасетами, для implicit-данных **не наблюдалось**, что чем больше истории взаимодействий у пользователя, тем точнее для него рекомендации — авторы предполагают, что это может быть связано с тем, что один телевизионный аккаунт часто смотрят сразу несколько разных людей (вся семья), из-за чего много данных не означает «более понятного» пользователя.

---

## 8. Итог и ограничения (что обсуждают авторы в конце)

Главный вывод статьи: implicit-наблюдения стоит явно разбивать на **два разных сигнала** — предпочтение (нравится/не нравится) и уверенность в этом предпочтении — и оптимизировать латентную факторную модель с учётом обоих сигналов сразу по **всем** парам пользователь-айтем, а не только по наблюдаемым. Ключевая инженерная находка — алгоритм ALS, который благодаря алгебраической структуре задачи считается за время, линейно растущее с объёмом данных, несмотря на то, что формально нужно учитывать миллиарды пар.

Авторы также честно обозначают ограничения и направления для дальнейшей работы:
- В модели все пары с нулевым предпочтением получают **одинаковую** минимальную уверенность $c_{ui}=1$, хотя на самом деле причины отсутствия взаимодействия разные (не знал о товаре / был занят / товар был недоступен) и потенциально заслуживают разных уровней уверенности — но такое усложнение увеличило бы вычислительные затраты.
- Стандартная схема «обучение на прошлом, тест на будущем» хорошо измеряет способность **предсказывать** поведение пользователя, но это не совсем то же самое, что цель настоящей рекомендательной системы — **предлагать пользователю что-то новое**, что он бы сам не нашёл. Оценить именно эту (более важную с бизнес-точки зрения) способность гораздо сложнее, и авторы предлагают лишь частичное решение — убирать из тестовой выборки уже просмотренные передачи, чтобы модель оценивалась именно на предсказании *новых* для пользователя интересов.

---

## Мини-глоссарий терминов

| Термин | Простое объяснение |
|---|---|
| **Collaborative Filtering (CF)** | Подход к рекомендациям, основанный только на истории поведения пользователей, без использования содержания товаров (жанр, описание и т.п.) |
| **Explicit feedback** | Пользователь сам явно оценивает объект (рейтинг, лайк) |
| **Implicit feedback** | Мы наблюдаем поведение (клики, покупки, просмотры) и косвенно судим о предпочтениях |
| **Latent factor model** | Модель, представляющая пользователей и айтемы в виде компактных числовых векторов ("скрытых факторов") |
| **Матричная факторизация** | Приближение большой разреженной матрицы взаимодействий произведением двух маленьких матриц |
| **Confidence (уверенность)** $c_{ui}$ | Насколько мы доверяем тому, что наблюдение $p_{ui}$ отражает реальное предпочтение |
| **ALS (Alternating Least Squares)** | Метод оптимизации, при котором поочерёдно фиксируется одна из двух матриц и аналитически пересчитывается другая |
| **Cold start (холодный старт)** | Проблема: для нового пользователя/товара без истории взаимодействий нет данных для рекомендации |
